## Analysis

This notebook analyses the performance of the graph-based surrogate model against ground-truth simulation. Optionally evaluates teacher-forced and/or bootstrapped predictions to inspect drift and stability over entire simulation.

In [ ]:
import pandas as pd
from pathlib import Path
from plots import plot_compare
from ml import calculate_residuals
from run_surrogate import load_graph_data, get_device, load_trained_model, run_all_predictions
from animate import animate_residuals
from run_surrogate import bootstrap_simulation

# Case Parameters
SIM_NAME = "example-model4"
CASE = "case0"

# Pathing
ROOT = Path().resolve().parent
BASE = ROOT / "sims" / SIM_NAME / "ml_training" / "graph" / CASE
MODEL_PATH =  BASE / "model_graphsage.pt"

# Animation Parameters
FPS = 20
CMAP = "turbo"
OFF_SCREEN = True
WINDOW_SIZE = (1920, 1088)

#TODO: Add heatmap (u vs p vs rmse) averaged over all epochs

#### Load Simulation Results (Truth)

In [ ]:
# Load PyFR simulation data
results_path = ROOT / "sims" / SIM_NAME / "ml_training" / f"{CASE}-results.csv"

df_sim = pd.read_csv(results_path)
df_sim = df_sim[df_sim['step'] > 0] # First time-step is initial condition for surrogate
df_sim.head(2)

#### Next-Step Prediction Test

Perform teacher-forced evaluation where the model iteratively predict the next 
state of each node using ground-truth (simulated) values as input at each step. 
This kind of run evaluates the model's "next-step" predictive accuracy without 
compounding its own errors and writes out node-level predictions for each step 
in a CSV with the standard schema for further analysis.

In [ ]:
# Loading GNN model
data = load_graph_data(SIM_NAME, CASE)
device = get_device()
model = load_trained_model(MODEL_PATH, device=device)

In [ ]:
# Run surrogate model
steps = df_sim['step'].max()  # Use the same number of steps as simulation
run_all_predictions(model, data, steps=steps, start_step=0)

In [ ]:
# Load surrogate prediction results
out_dir = ROOT / "sims" / SIM_NAME / "ml_training" / "rollouts"
out_csv_path = out_dir / f"{CASE}-teacher_forced.csv"
df_ml = pd.read_csv(out_csv_path)
df_ml.head(2)

In [ ]:
# Quick checks
assert len(df_ml) != 0, "Missing surrogate data!"
assert len(df_sim) != 0, "Missing sim data!"
assert len(df_ml) == len(df_sim), f"Should be equal: {len(df_ml)}, {len(df_sim)}"

In [ ]:
# Calculate residuals at each node
df_res = calculate_residuals(df_sim, df_ml)
df_res.describe()

In [ ]:
# Quick checks
assert len(df_ml) == len(df_sim) == len(df_res), f"Should be equal: {len(df_ml)}, {len(df_sim)}, {len(df_res)}"

In [ ]:
"""
[ Nodal Residual Analysis ]

This section analyses how well the surrogate (ML) model predicts the simulation 
at each node and time step.

The plot generated below shows, for a selected time step (STEP), a comparison 
between the simulation ("ground truth"), the surrogate model's prediction, and 
the residual (error) between them, at all nodes. 

The visualisation helps diagnose:
- Where in the domain the model performs well or poorly,
- Which regions tend to have larger errors,
- And whether the errors display any spatial pattern or structure.
"""

# Plot single step: 
STEP = 800 # <-- manually select

fig = plot_compare(
    df_sim, df_ml, df_res, 
    metric="p",                # p, u, v, vn
    step=STEP, 
    plot_type="contour"        # "contour" or "scatter"
)
fig.show()

In [ ]:
"""
If enabled, create an animation of the residuals between simulation and 
surrogate model predictions at all time steps
"""
ANIMATE = True

if ANIMATE:
    animate_residuals(
        df_sim, df_ml, df_res, 
        sim_name=SIM_NAME, 
        dir_name="next-step-residuals", 
        window_size=(900, 1260), 
        plot_type="contour"
    )

#### Boostrap Analaysis

The bootstrap model is a variant of the surrogate (ML) model designed to assess 
the stability and predictive reliability of the surrogate on its own outputs. 
In the bootstrap procedure, the surrogate model is used recursively and 
predictions at one time step are used as inputs to predict the next time step, 
simulating a full rollout where only the surrogate's outputs are used 
(without correction from the 'ground truth' simulation data).

This approach evaluates how errors may accumulate over time when the surrogate 
is applied in a self-consistent way, mirroring how it would be used in practice 
during deployment. It also highlights any drift, instability, or significant 
error amplification present when the surrogate is run in closed-loop fashion.

In [ ]:
"""
NOTE: Comparing the surrogate solely via step-by-step residuals can be 
misleading. Should consider statistical evaluation methods (e.g. RMSE 
distributions) to better assess surrogate performance over time.
"""
steps = df_sim['step'].max()  # Use the same number of steps as simulation\
# steps = 400
br_path = bootstrap_simulation(SIM_NAME, CASE, steps=steps, start_step=0)

In [ ]:
# Load bootstrap results
out_dir = ROOT / "sims" / SIM_NAME / "ml_training" / "bootstrapped"
out_csv_path = out_dir / f"{CASE}-bootstrap.csv"
df_boot = pd.read_csv(out_csv_path)
df_boot.head(2)

In [ ]:
# Calculate residuals
df_boot_res = calculate_residuals(df_sim, df_boot)
df_boot_res.describe()

In [ ]:
# Quick checks
assert len(df_boot) == len(df_sim) == len(df_boot_res), f"Should be equal: {len(df_boot)}, {len(df_sim)}, {len(df_boot_res)}"

In [ ]:
"""
Animate bootstrap residual plots

If enabled, create an animation of the residuals between simulation and 
surrogate model predictions at all time steps
"""
ANIMATE = True

if ANIMATE:
    animate_residuals(
        df_sim, df_boot, df_boot_res, 
        sim_name=SIM_NAME, 
        dir_name="boot-residuals", 
        window_size=(900, 1260), 
        plot_type="contour"
    )